# Community Structure Sweeps — CIC3 Metrics

Sweep the inter-community edge probability ($p_{\mathrm{inter}}$) of an SBM with
$K=10$ equal communities at three fixed intra-community probabilities
($p_{\mathrm{intra}}$). For each configuration, run $C$ competing CIC3 contagions
with random seeding and measure four metrics:

- **$A_g$ / $A_g^{td}$** — global attainment (undiscounted and time-discounted)
- **$D_g$** — deadweight (infections beyond quota)
- **$P_g$** — penetration (mean BFS hop depth from seeds)

For each metric we produce two plots:
1. **Individual** — one line per $p_{\mathrm{intra}}$ value
2. **Averaged** — mean over the three $p_{\mathrm{intra}}$ values

Attainment plots use the same $A_g^{td}$ / $A_g$ shading convention as
`cic3_attainment_sweep.ipynb`.

In [19]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import pickle
import hashlib
import os

from scm import SBMGenerator, MultiRandomSeeding
from scm.cic3_simulator import CIC3Simulator
from scm.analysis import (
    attainment,
    time_discounted_attainment,
    exponential_decay,
    deadweight,
    penetration,
)

# --- Cache helpers ---
CACHE_DIR = '../results/community_structure_sweeps'

def _param_fingerprint(**kwargs):
    raw = str(sorted(kwargs.items()))
    return hashlib.md5(raw.encode()).hexdigest()[:12]

def _cache_path(name):
    return os.path.join(CACHE_DIR, f'{name}.pkl')

def save_cache(name, obj, **fingerprint_kwargs):
    os.makedirs(CACHE_DIR, exist_ok=True)
    fp = _param_fingerprint(**fingerprint_kwargs)
    path = _cache_path(name)
    with open(path, 'wb') as f:
        pickle.dump({'fingerprint': fp, 'data': obj}, f)
    print(f'  Cached {name} -> {path} (fp={fp})')

def load_cache(name, **fingerprint_kwargs):
    path = _cache_path(name)
    if not os.path.exists(path):
        return None
    fp = _param_fingerprint(**fingerprint_kwargs)
    with open(path, 'rb') as f:
        blob = pickle.load(f)
    if blob['fingerprint'] != fp:
        print(f'  Cache stale for {name} (disk={blob["fingerprint"]}, current={fp})')
        return None
    print(f'  Loaded {name} from cache (fp={fp})')
    return blob['data']

## Parameters

In [ ]:
# Network
N = 2000
K_COMM = 10
COMMUNITY_SIZES = [N // K_COMM] * K_COMM  # [50]*10

# Sweep: inter-community edge probability
P_INTER_VALS = np.linspace(0.1, 1.0, 19)

# Fixed intra-community edge probabilities (3 values)
P_INTRA_VALS = [0.1, 0.3, 0.6]

# Triangle probability (intra only, uniform across communities)
P_TRI_INTRA = 0.05

# CIC3 parameters
C = 50
QUOTAS = [N // C] * C  # sum = N, no slack
SEEDS_PER_CONTAGION = 1
T_MAX = 2000
DECAY_RATE = 0.05
V = exponential_decay(DECAY_RATE)

# Infection parameters (rescaled infectivity)
LAMBDA = 1.0
LAMBDA_DELTA = 2.0

# Simulation
NUM_TRIALS = 15
TOPO_SEED = 2025

## Topology Generation

Generate one SBM per $(p_{\mathrm{inter}}, p_{\mathrm{intra}})$ pair.
Each has $K=10$ equal communities of size $N/K$.

In [ ]:
topologies = {}

for p_inter in P_INTER_VALS:
    for p_intra in P_INTRA_VALS:
        key = (round(p_inter, 4), p_intra)
        block_matrix = np.full((K_COMM, K_COMM), p_intra)
        np.fill_diagonal(block_matrix, p_inter)

        gen = SBMGenerator(
            community_sizes=COMMUNITY_SIZES,
            block_matrix=block_matrix,
            triangle_block_probs=[P_TRI_INTRA] * K_COMM,
        )
        links, triangles = gen.generate(seed=TOPO_SEED)
        topologies[key] = {
            'links': links,
            'triangles': triangles,
            'N': gen.N,
            'k_avg': gen.k_avg,
            'k_d_avg': gen.k_delta_avg,
        }
        print(f'  p_inter={p_inter:.2f} p_intra={p_intra:.3f}'
              f' -> k_avg={gen.k_avg:.2f} k_d_avg={gen.k_delta_avg:.2f}')

print(f'\nGenerated {len(topologies)} topologies')

## Run CIC3 Simulations

For each topology, run `NUM_TRIALS` independent CIC3 simulations with random
seeding. Record all four metrics: $A_g$, $A_g^{td}$, $D_g$, $P_g$.

In [ ]:
def run_community_sweep():
    fp_kwargs = dict(
        N=N, K=K_COMM, p_inters=list(P_INTER_VALS),
        p_intras=P_INTRA_VALS, p_tri=P_TRI_INTRA,
        C=C, quotas=QUOTAS, t_max=T_MAX,
        decay_rate=DECAY_RATE, num_trials=NUM_TRIALS,
        topo_seed=TOPO_SEED, lam=LAMBDA, lam_delta=LAMBDA_DELTA,
    )
    cached = load_cache('community_sweep', **fp_kwargs)
    if cached is not None:
        return cached

    results = {}

    for p_inter in P_INTER_VALS:
        for p_intra in P_INTRA_VALS:
            key = (round(p_inter, 4), p_intra)
            topo = topologies[key]
            k_avg = topo['k_avg']
            k_d_avg = topo['k_d_avg']

            beta = LAMBDA / k_avg
            beta_delta = LAMBDA_DELTA / k_d_avg
            betas = [beta] * C
            beta_deltas = [beta_delta] * C

            num_seeds_per_contagion = [SEEDS_PER_CONTAGION] * C #[max(1, q // 5) for q in QUOTAS]

            Ag_trials, Ag_td_trials = [], []
            Dg_trials, Pg_trials = [], []

            for trial in range(NUM_TRIALS):
                seeder = MultiRandomSeeding(
                    N=topo['N'],
                    num_seeds_per_contagion=num_seeds_per_contagion,
                    links=topo['links'],
                    triangles=topo['triangles'],
                )
                seeds = seeder.seed()

                sim = CIC3Simulator(
                    links=topo['links'],
                    triangles=topo['triangles'],
                    initial_infected_per_contagion=seeds,
                    betas=betas,
                    beta_deltas=beta_deltas,
                    quotas=QUOTAS,
                    stop_on_all_quotas_met=False,
                )
                sim.run(T_MAX)

                Ai, Ag = attainment(sim.infected_by, QUOTAS)
                Ai_td, Ag_td = time_discounted_attainment(
                    sim.infected_by, sim.infection_times, QUOTAS, V
                )
                Di, Dg = deadweight(sim.infected_by, QUOTAS)
                Pi, Pg = penetration(topo['links'], sim.infected_by, seeds)

                Ag_trials.append(Ag)
                Ag_td_trials.append(Ag_td)
                Dg_trials.append(Dg)
                Pg_trials.append(Pg)

            results[key] = {
                'Ag_mean': np.mean(Ag_trials),
                'Ag_std': np.std(Ag_trials),
                'Ag_td_mean': np.mean(Ag_td_trials),
                'Ag_td_std': np.std(Ag_td_trials),
                'Dg_mean': np.mean(Dg_trials),
                'Dg_std': np.std(Dg_trials),
                'Pg_mean': np.mean(Pg_trials),
                'Pg_std': np.std(Pg_trials),
            }
            print(f'  p_inter={p_inter:.2f} p_intra={p_intra:.3f}:'
                  f' Ag_td={results[key]["Ag_td_mean"]:.3f}'
                  f' Dg={results[key]["Dg_mean"]:.1f}'
                  f' Pg={results[key]["Pg_mean"]:.2f}')

    save_cache('community_sweep', results, **fp_kwargs)
    return results

all_results = run_community_sweep()
print(f'\nDone — {len(all_results)} configurations')

## Attainment Plots

Two figures:
1. **Individual** — one line per $p_{\mathrm{intra}}$, showing $A_g^{td}$ (solid) and $A_g$ (faint) with shading between them.
2. **Averaged** — mean over $p_{\mathrm{intra}}$ values, with the same $A_g$ / $A_g^{td}$ shading.

In [ ]:
p_intra_colors = {0.1: 'blue', 0.3: 'green', 0.6: 'red'}

fig, ax = plt.subplots(figsize=(8, 5.5))

for p_intra in P_INTRA_VALS:
    color = p_intra_colors[p_intra]
    Ag_means, Ag_td_means = [], []
    Ag_stds, Ag_td_stds = [], []

    for p_inter in P_INTER_VALS:
        key = (round(p_inter, 4), p_intra)
        res = all_results[key]
        Ag_means.append(res['Ag_mean'])
        Ag_td_means.append(res['Ag_td_mean'])
        Ag_stds.append(res['Ag_std'])
        Ag_td_stds.append(res['Ag_td_std'])

    Ag_means = np.array(Ag_means)
    Ag_td_means = np.array(Ag_td_means)
    Ag_stds = np.array(Ag_stds)
    Ag_td_stds = np.array(Ag_td_stds)

    # Faint: undiscounted A_g
    ax.plot(P_INTER_VALS, Ag_means, color=color, lw=1.5, alpha=0.35,
            label=rf'$A_g$ ($p_{{inter}}={p_intra}$)')
    ax.fill_between(P_INTER_VALS, Ag_means - Ag_stds, Ag_means + Ag_stds,
                     color=color, alpha=0.08)

    # Solid: time-discounted A_g^td
    ax.plot(P_INTER_VALS, Ag_td_means, color=color, lw=2.5,
            label=rf'$A_g^{{td}}$ ($p_{{inter}}={p_intra}$)')
    ax.fill_between(P_INTER_VALS, Ag_td_means - Ag_td_stds,
                     Ag_td_means + Ag_td_stds, color=color, alpha=0.12)

    # Shade between A_g and A_g^td
    ax.fill_between(P_INTER_VALS, Ag_td_means, Ag_means,
                     color=color, alpha=0.15)

ax.set_xlim(P_INTER_VALS[0], P_INTER_VALS[-1])
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel(r'Inter-community edge probability, $p_{\mathrm{inter}}$', fontsize=12)
ax.set_ylabel(r'Global attainment, $A_g^{td}$ / $A_g$', fontsize=12)
ax.set_title(
    rf'CIC3 Attainment vs Community Structure  '
    rf'($N={N}$, $K={K_COMM}$, $C={C}$, '
    rf'$\lambda={LAMBDA}$, $\lambda_\Delta={LAMBDA_DELTA}$)',
    fontsize=12,
)
ax.legend(loc='lower right', framealpha=1, edgecolor='black', fontsize=8)
ax.grid(alpha=0.3)
plt.tight_layout()
out_path = '../figures/community_sweep_attainment_individual.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

In [ ]:
# Average over p_intra values
Ag_stack = np.array([
    [all_results[(round(p_inter, 4), p_intra)]['Ag_mean']
     for p_inter in P_INTER_VALS]
    for p_intra in P_INTRA_VALS
])
Ag_td_stack = np.array([
    [all_results[(round(p_inter, 4), p_intra)]['Ag_td_mean']
     for p_inter in P_INTER_VALS]
    for p_intra in P_INTRA_VALS
])
Ag_avg = Ag_stack.mean(axis=0)
Ag_td_avg = Ag_td_stack.mean(axis=0)

fig, ax = plt.subplots(figsize=(8, 5.5))

# Shaded gap between A_g and A_g^td
ax.fill_between(P_INTER_VALS, Ag_td_avg, Ag_avg,
                color='orange', alpha=0.25, label='Time-discount gap')

# A_g line
ax.plot(P_INTER_VALS, Ag_avg, color='steelblue', lw=2.5, label=r'$A_g$')

# A_g^td line
ax.plot(P_INTER_VALS, Ag_td_avg, color='darkorange', lw=2.5,
        label=r'$A_g^{td}$')

ax.set_xlim(P_INTER_VALS[0], P_INTER_VALS[-1])
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel(r'Inter-community edge probability, $p_{\mathrm{inter}}$', fontsize=12)
ax.set_ylabel(r'Global attainment, $A_g^{td}$ / $A_g$', fontsize=12)
ax.set_title(
    rf'CIC3 Attainment vs Community Structure (avg over $p_{{inter}}$)  '
    rf'($N={N}$, $K={K_COMM}$, $C={C}$, '
    rf'$\lambda={LAMBDA}$, $\lambda_\Delta={LAMBDA_DELTA}$)',
    fontsize=12,
)
ax.legend(loc='lower right', framealpha=1, edgecolor='black', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
out_path = '../figures/community_sweep_attainment_averaged.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

## Deadweight Plots

$D_g$ = mean number of excess infections beyond quota per contagion.
Higher community structure (larger $p_{\mathrm{inter}}$) should reduce deadweight
because infections stay contained within communities.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

for p_intra in P_INTRA_VALS:
    color = p_intra_colors[p_intra]
    Dg_means, Dg_stds = [], []

    for p_inter in P_INTER_VALS:
        key = (round(p_inter, 4), p_intra)
        res = all_results[key]
        Dg_means.append(res['Dg_mean'])
        Dg_stds.append(res['Dg_std'])

    Dg_means = np.array(Dg_means)
    Dg_stds = np.array(Dg_stds)

    ax.plot(P_INTER_VALS, Dg_means, color=color, lw=2.5,
            label=rf'$D_g$ ($p_{{inter}}={p_intra}$)')
    ax.fill_between(P_INTER_VALS, Dg_means - Dg_stds, Dg_means + Dg_stds,
                     color=color, alpha=0.12)

ax.set_xlim(P_INTER_VALS[0], P_INTER_VALS[-1])
ax.set_xlabel(r'Inter-community edge probability, $p_{\mathrm{inter}}$', fontsize=12)
ax.set_ylabel(r'Deadweight, $D_g$', fontsize=12)
ax.set_title(
    rf'CIC3 Deadweight vs Community Structure  '
    rf'($N={N}$, $K={K_COMM}$, $C={C}$, '
    rf'$\lambda={LAMBDA}$, $\lambda_\Delta={LAMBDA_DELTA}$)',
    fontsize=12,
)
ax.legend(loc='best', framealpha=1, edgecolor='black', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
out_path = '../figures/community_sweep_deadweight_individual.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

In [ ]:
Dg_stack = np.array([
    [all_results[(round(p_inter, 4), p_intra)]['Dg_mean']
     for p_inter in P_INTER_VALS]
    for p_intra in P_INTRA_VALS
])
Dg_avg = Dg_stack.mean(axis=0)

fig, ax = plt.subplots(figsize=(8, 5.5))

ax.plot(P_INTER_VALS, Dg_avg, color='darkred', lw=2.5, label=r'$D_g$ (avg)')
ax.fill_between(P_INTER_VALS, Dg_avg - Dg_stack.std(axis=0),
                Dg_avg + Dg_stack.std(axis=0), color='darkred', alpha=0.12)

ax.set_xlim(P_INTER_VALS[0], P_INTER_VALS[-1])
ax.set_xlabel(r'Inter-community edge probability, $p_{\mathrm{inter}}$', fontsize=12)
ax.set_ylabel(r'Deadweight, $D_g$', fontsize=12)
ax.set_title(
    rf'CIC3 Deadweight vs Community Structure (avg over $p_{{inter}}$)  '
    rf'($N={N}$, $K={K_COMM}$, $C={C}$, '
    rf'$\lambda={LAMBDA}$, $\lambda_\Delta={LAMBDA_DELTA}$)',
    fontsize=12,
)
ax.legend(loc='best', framealpha=1, edgecolor='black', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
out_path = '../figures/community_sweep_deadweight_averaged.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

## Penetration Plots

$P_g$ = mean BFS hop distance from seeds through each contagion's subgraph.
Higher community structure (larger $p_{\mathrm{inter}}$) may increase penetration
depth within communities but reduce cross-community spread.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))

for p_intra in P_INTRA_VALS:
    color = p_intra_colors[p_intra]
    Pg_means, Pg_stds = [], []

    for p_inter in P_INTER_VALS:
        key = (round(p_inter, 4), p_intra)
        res = all_results[key]
        Pg_means.append(res['Pg_mean'])
        Pg_stds.append(res['Pg_std'])

    Pg_means = np.array(Pg_means)
    Pg_stds = np.array(Pg_stds)

    ax.plot(P_INTER_VALS, Pg_means, color=color, lw=2.5,
            label=rf'$P_g$ ($p_{{inter}}={p_intra}$)')
    ax.fill_between(P_INTER_VALS, Pg_means - Pg_stds, Pg_means + Pg_stds,
                     color=color, alpha=0.12)

ax.set_xlim(P_INTER_VALS[0], P_INTER_VALS[-1])
ax.set_xlabel(r'Inter-community edge probability, $p_{\mathrm{inter}}$', fontsize=12)
ax.set_ylabel(r'Penetration, $P_g$', fontsize=12)
ax.set_title(
    rf'CIC3 Penetration vs Community Structure  '
    rf'($N={N}$, $K={K_COMM}$, $C={C}$, '
    rf'$\lambda={LAMBDA}$, $\lambda_\Delta={LAMBDA_DELTA}$)',
    fontsize=12,
)
ax.legend(loc='best', framealpha=1, edgecolor='black', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
out_path = '../figures/community_sweep_penetration_individual.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

In [ ]:
Pg_stack = np.array([
    [all_results[(round(p_inter, 4), p_intra)]['Pg_mean']
     for p_inter in P_INTER_VALS]
    for p_intra in P_INTRA_VALS
])
Pg_avg = Pg_stack.mean(axis=0)

fig, ax = plt.subplots(figsize=(8, 5.5))

ax.plot(P_INTER_VALS, Pg_avg, color='darkgreen', lw=2.5, label=r'$P_g$ (avg)')
ax.fill_between(P_INTER_VALS, Pg_avg - Pg_stack.std(axis=0),
                Pg_avg + Pg_stack.std(axis=0), color='darkgreen', alpha=0.12)

ax.set_xlim(P_INTER_VALS[0], P_INTER_VALS[-1])
ax.set_xlabel(r'Inter-community edge probability, $p_{\mathrm{inter}}$', fontsize=12)
ax.set_ylabel(r'Penetration, $P_g$', fontsize=12)
ax.set_title(
    rf'CIC3 Penetration vs Community Structure (avg over $p_{{inter}}$)  '
    rf'($N={N}$, $K={K_COMM}$, $C={C}$, '
    rf'$\lambda={LAMBDA}$, $\lambda_\Delta={LAMBDA_DELTA}$)',
    fontsize=12,
)
ax.legend(loc='best', framealpha=1, edgecolor='black', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
out_path = '../figures/community_sweep_penetration_averaged.png'
plt.savefig(out_path, dpi=300, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()